# 01 · Run the frozen support benchmark

Run All starts the 20-seed confirmation by default. Each seed runs driver-switch, constant and nuisance-switch cases. The learned, dense and shuffled arms share a frozen readout, origin grid and tuned selector. Direct history supplies a forecast baseline.

For a short check set `PROFILE = "smoke"` in this notebook and notebook 02. Adjust `WORKERS` for the machine. Reuse the same run ID to resume a matching run.


In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select the project Python 3.14 .venv kernel.")

working_directory = Path.cwd().resolve()
try:
    REPO = next(
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "effectome").is_dir()
    )
except StopIteration as error:
    raise RuntimeError("Open this notebook from inside the effectome application repository.") from error

project_src = str(REPO / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

print("Notebook kernel", sys.executable)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
PROJECT_ROOT = REPO


In [ ]:
from pathlib import Path

ROOT = PROJECT_ROOT
PROFILE = "confirmation"  # Set "smoke" here and in the other run notebook for a short check.
RUN_ID = "support-01"

if PROFILE not in {"smoke", "confirmation"}:
    raise ValueError("PROFILE must be smoke or confirmation.")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("Choose a single directory name for RUN_ID.")
RUN_ROOT = ROOT / "outputs" / "effectome-rollout-support" / PROFILE / RUN_ID
print("Run directory", RUN_ROOT)


In [ ]:
import os
import signal
import subprocess
import sys

RUN = True
WORKERS = 3

command = [
    sys.executable, "-u", "-m", "synthetic.behavior_rollout_support", "--profile", PROFILE,
    "--workers", str(WORKERS), "--output-root", str(RUN_ROOT),
]
if RUN:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    with (RUN_ROOT / "run.log").open("a", encoding="utf-8") as log:
        with subprocess.Popen(
            [*command, "--run"], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, start_new_session=(os.name == "posix"),
        ) as process:
            try:
                for line in process.stdout:
                    log.write(line)
                    log.flush()
                    print(line, end="", flush=True)
                if process.wait() != 0:
                    raise subprocess.CalledProcessError(process.returncode, process.args)
            except KeyboardInterrupt:
                if os.name == "posix":
                    os.killpg(process.pid, signal.SIGTERM)
                else:
                    process.terminate()
                raise
else:
    status = subprocess.check_output([*command, "--status"], cwd=ROOT, text=True)
    print(status)


Watch `support-report.json` or `run.log` in the printed run directory for progress. Continue to notebook 02 once the report records `execution_complete: true`.
